# Support Ticket Intelligence Agent — Demo

A small multi-stage LLM pipeline that takes raw, unstructured customer support tickets and:

1. **Summarizes** each ticket into a structured form (summary, key issues, sentiment, action items)
2. **Evaluates** that summary using an LLM-as-judge (factual accuracy, completeness, conciseness, actionability, tone)
3. **Generates** a policy-grounded, customer-facing email response
4. **Evaluates** that email response using a second LLM-as-judge (policy alignment, scope check, grounding, tone, actionability)

This notebook runs the pipeline end-to-end on a small synthetic ticket dataset (`data/sample_tickets.csv`) for demo purposes.
The full pipeline logic lives in `src/`; this notebook just calls into it and displays results.

> This project was built independently as a portfolio piece, inspired by a coursework assignment on AI agents in business applications. The dataset, company name, and policies here are synthetic.

## Setup

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.pipeline import run_pipeline

## Run the pipeline

This runs all four stages (summarize → evaluate → respond → evaluate) over every row in the sample dataset and writes a compiled CSV to `outputs/`.

In [ ]:
results_df = run_pipeline(input_csv="../data/sample_tickets.csv", output_csv="../outputs/compiled_ticket_outputs.csv")
results_df.head()

## Inspect a single ticket end-to-end

In [ ]:
row = results_df.iloc[0]

print("RAW TICKET")
print(row["ticket_text"])
print("\nSTRUCTURED SUMMARY")
print(f"Summary: {row['summary']}")
print(f"Sentiment: {row['sentiment']}")
print(f"Key Issues: {row['key_issues']}")
print(f"Action Items: {row['action_items']}")
print("\nSUMMARY EVAL")
print(f"Overall Score: {row['overall_score']} — {row['justification']}")
print("\nGENERATED EMAIL RESPONSE")
print(row["email_response"])
print("\nRESPONSE EVAL")
print(f"Passed: {row['email_eval_passed']} | Recommendation: {row['email_eval_recommendation']}")
print(f"Scope Check: {row['email_eval_scope_check_score']} — {row['email_eval_scope_check_justification']}")

## Notes on this run

Worth calling out: one ticket in the sample set (a food-delivery complaint unrelated to this company) is intentionally
included to test the **scope check** built into the response evaluator. A naive pipeline will happily draft an apology
and a refund offer for a completely unrelated company's mistake — the judge stage is what catches that before a human
agent would ever see it. See the README for the full writeup of this failure mode and the fix.